# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya Exploration with `mlcroissant`
This notebook provides a full, template-based guide for loading and exploring the FAIR² Mental Health Survey Dataset from Kilifi County, Kenya using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via the Croissant schema URL below:

`https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json`

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

The dataset is defined with a Croissant schema (JSON-LD), so each entity (record set, field, column) must be referenced by its `@id`.

Let's explore which record sets are available, and their fields, using their `@id`.

In [ ]:
# Explore available record sets and fields
record_sets = list(dataset.record_set_ids)
print(f"Available record sets @id:")
for rs_id in record_sets:
    print(f"  - {rs_id}")
    record_set_obj = dataset.get_record_set(rs_id)
    fields = [f['@id'] for f in record_set_obj['fields']] if 'fields' in record_set_obj else []
    print(f"    Fields @id: {fields}")

# For demonstration, show the first few records from the first available record set
if record_sets:
    rs_id = record_sets[0]
    for i, record in enumerate(dataset.records(record_set=rs_id)):
        if i < 3:
            print(f"Record {i+1}: {record}")

## 3. Data Extraction
Load data from one or more record sets into DataFrames for analysis. Use record set and field `@id` references as discovered above.

For this example, let's extract records from all available record sets.

In [ ]:
# Extract data from each record set
dataframes = {}
for rs_id in record_sets:
    records = list(dataset.records(record_set=rs_id))
    df = pd.DataFrame(records)
    dataframes[rs_id] = df
    print(f"Columns in record set {rs_id}: {df.columns.tolist()}")
    display(df.head())

# For further steps, focus on the first record set
main_rs_id = record_sets[0] if record_sets else None

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

**Note:** All fields are referenced by their `@id`.

Let's process numeric survey scores (e.g., PHQ-9, GAD-7, PSQ) and demographic fields.

In [ ]:
# Example: Filter and normalize PHQ-9 score (ensure to use the proper @id!)

df = dataframes[main_rs_id]

# Find the field IDs for PHQ-9, GAD-7, PSQ scores
phq9_id = None
gad7_id = None
psq_id = None
education_id = None

record_set_obj = dataset.get_record_set(main_rs_id)
if 'fields' in record_set_obj:
    for field in record_set_obj['fields']:
        label = field.get('label', '').lower()
        if 'phq-9' in label:
            phq9_id = field['@id']
        elif 'gad-7' in label:
            gad7_id = field['@id']
        elif 'psq' in label:
            psq_id = field['@id']
        elif 'education' in label:
            education_id = field['@id']

# Use PHQ-9 score if available; else use GAD-7 or PSQ
chosen_numeric_id = phq9_id or gad7_id or psq_id

if chosen_numeric_id and chosen_numeric_id in df.columns:
    # Threshold for filtering high PHQ-9 scores
    threshold = 10
    filtered_df = df[df[chosen_numeric_id] > threshold]
    print(f"Filtered records with {chosen_numeric_id} > {threshold}:")
    display(filtered_df.head())

    # Normalize the score
    filtered_df[f"{chosen_numeric_id}_normalized"] = (
        filtered_df[chosen_numeric_id] - filtered_df[chosen_numeric_id].mean()
    ) / filtered_df[chosen_numeric_id].std()
    print(f"Normalized {chosen_numeric_id} for filtered records:")
    display(filtered_df[[chosen_numeric_id, f"{chosen_numeric_id}_normalized"]].head())

    # Group by Education if available
    group_field = education_id
    if group_field and group_field in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field)[chosen_numeric_id].mean().reset_index()
        print(f"Grouped data by {group_field} (mean {chosen_numeric_id}):")
        display(grouped_df.head())
else:
    print("Numeric survey score field (PHQ-9, GAD-7, or PSQ) not found in columns.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

Here we will plot a histogram of the selected survey score and visualize the relationship with education level if possible.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of survey scores
if chosen_numeric_id and chosen_numeric_id in df.columns:
    plt.figure(figsize=(8,4))
    sns.histplot(df[chosen_numeric_id].dropna(), bins=20, kde=True)
    plt.title(f'Distribution of {chosen_numeric_id}')
    plt.xlabel('Score')
    plt.ylabel('Frequency')
    plt.show()

    # Boxplot by education level
    if education_id and education_id in df.columns:
        plt.figure(figsize=(10,5))
        sns.boxplot(x=df[education_id], y=df[chosen_numeric_id])
        plt.title(f'{chosen_numeric_id} by {education_id}')
        plt.xlabel('Education Level')
        plt.ylabel('Survey Score')
        plt.show()


## 6. Conclusion
In this notebook, we used the `mlcroissant` library to load and explore the FAIR² Mental Health Survey Dataset from Kilifi County, Kenya. We reviewed available record sets and their field `@id`s, loaded data into DataFrames, performed basic filtering and normalization, grouped data by demographic attributes, and visualized survey score distributions.

Key findings:
- The dataset contains rich survey information including PHQ-9, GAD-7, and PSQ scores referenced by their `@id`s.
- High scores (e.g., PHQ-9 > 10) can be filtered and analyzed in relation to demographic factors such as education level.
- Visualization reveals the score distribution and group-level trends.

Further exploration could include time-based analyses, more detailed outlier detection, and integration with public health data for broader context.